In [ ]:
###### This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

The problem is figuring out a way to make a machine learning model play Tetris for a long time.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

In [1]:
import sys

print(sys.version)
print(sys.executable)

# !{sys.executable} -m venv ~/tetris_env
# !~/tetris_env/bin/python -m pip install --upgrade pip
# !~/tetris_env/bin/python -m pip install ipykernel
# !~/tetris_env/bin/python -m ipykernel install --user --name=tetris_env --display-name="Python (Tetris)"
# !git clone https://gitlab.com/yukiman/tetris_a.git
# %cd tetris_a
# !ls -la
# !~/tetris_env/bin/python -m pip install -r requirements.txt
# !{sys.executable} -m pip install -r requirements.txt

3.12.6 | packaged by conda-forge | (main, Sep 22 2024, 14:16:49) [GCC 13.3.0]
/home/jupyter-250661/tetris_env/bin/python


In [3]:
!jupyter kernelspec list

Available kernels:
  tetris_env    /home/jupyter-250661/.local/share/jupyter/kernels/tetris_env
  python3       /opt/tljh/user/share/jupyter/kernels/python3


# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

A virtual environment for the Tetris game was created, in addition to a new kernel, Python (Tetris).

# 4.Prepare the Data


Apply any data transformations and explain what and why


In [18]:
import numpy as np
import random
import copy
import sys
#!{sys.executable} -m pip install matplotlib
from copy import deepcopy
import matplotlib.pyplot as plt
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
from board import Board
from piece import Piece, BODIES

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

board = Board()
piece = Piece()
print("Board dimensions:", board.width, "x", board.height)
print("Piece body:", piece.body)
print("Piece skirt:", piece.skirt)

PyTorch: 2.15.0+cu132
CUDA available: False
Using device: cpu
Board dimensions: 10 x 20
Piece body: ((0, 0), (1, 0), (1, 1), (1, 2))
Piece skirt: [0, 0]


In [2]:
%cd src/

/home/jupyter-250661/Project_6/tetris_a/src


# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [19]:
import sys
sys.path.insert(0, "src")

import random
from collections import deque
from copy import deepcopy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from board import Board
from piece import Piece

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


def extract_afterstate_features(board, lines_cleared):
    """
    Extracts 6 clean, normalized features from a board AFTER a piece has been
    placed and full rows have been cleared.
    """
    width = board.width
    height = board.height  # 20
    heights = board.heights

    # 1. Aggregate height (normalized by max possible cells: 200)
    agg_height = sum(heights)

    # 2. Max column height (normalized by 20)
    max_height = max(heights) if heights else 0

    # 3. Total holes (empty cells with at least one filled cell above them)
    holes = 0
    for x in range(width):
        for y in range(heights[x]):
            if not board.board[y][x]:
                holes += 1

    # 4. Bumpiness (sum of absolute height differences between adjacent columns)
    bumpiness = sum(abs(heights[x] - heights[x + 1]) for x in range(width - 1))

    # 5. Deep wells (sum of depth of 1-wide wells bounded on both sides)
    wells = 0
    for x in range(width):
        left_h = heights[x - 1] if x > 0 else height
        right_h = heights[x + 1] if x < width - 1 else height
        min_neighbor = min(left_h, right_h)
        if min_neighbor > heights[x]:
            wells += (min_neighbor - heights[x])

    return np.array([
        agg_height / 200.0,
        max_height / 20.0,
        holes / 200.0,
        bumpiness / 180.0,
        wells / 200.0,
        lines_cleared / 4.0
    ], dtype=np.float32)


def get_possible_afterstates(board, piece):
    """
    Generates all valid (rotation_idx, x, rotated_piece, afterstate_features, lines_cleared, done)
    for the current piece without mutating the original board.
    """
    candidates = []
    current = piece
    seen_bodies = set()

    for r in range(4):
        # Deduplicate symmetric rotations (e.g., Square or Stick)
        body_key = tuple(sorted(current.body))
        if body_key not in seen_bodies:
            seen_bodies.add(body_key)

            for x in range(board.width):
                if x + len(current.skirt) > board.width:
                    continue

                y = board.drop_height(current, x)
                if y < 0 or y + max(pos[1] for pos in current.body) >= board.height + 4:
                    continue

                # Simulate placement on a lightweight copy
                sim_board = Board()
                sim_board.board = [row[:] for row in board.board]
                sim_board.widths = board.widths[:]
                sim_board.heights = board.heights[:]

                res = sim_board.place(x, y, current)
                if isinstance(res, Exception):
                    continue

                lines = sim_board.clear_rows()
                done = sim_board.top_filled()
                features = extract_afterstate_features(sim_board, lines)

                candidates.append({
                    "x": x,
                    "piece": current,
                    "features": features,
                    "lines": lines,
                    "done": done
                })

        current = current.get_next_rotation()

    return candidates

Using device: cpu


In [20]:
class TetrisAfterstateEnv:
    def __init__(self):
        self.board = None
        self.piece = None
        self.pieces_placed = 0
        self.rows_cleared = 0

    def reset(self):
        self.board = Board()
        self.piece = Piece()
        self.pieces_placed = 0
        self.rows_cleared = 0
        return get_possible_afterstates(self.board, self.piece)

    def step(self, chosen_move):
        x = chosen_move["x"]
        piece = chosen_move["piece"]
        y = self.board.drop_height(piece, x)

        res = self.board.place(x, y, piece)
        if isinstance(res, Exception):
            raise ValueError("Invalid placement chosen")

        self.pieces_placed += 1
        lines = self.board.clear_rows()
        self.rows_cleared += lines
        done = self.board.top_filled()

        # Clean reward: survival bonus + quadratic bonus for multi-line clears
        reward = 1.0 + (lines ** 2) * 10.0
        if done:
            reward = -10.0

        self.piece = Piece()
        next_candidates = [] if done else get_possible_afterstates(self.board, self.piece)
        if not next_candidates:
            done = True

        return next_candidates, reward, done


class AfterstateDQN(nn.Module):
    def __init__(self, input_dim=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


class AfterstateAgent:
    def __init__(self):
        self.policy_net = AfterstateDQN().to(device)
        self.target_net = AfterstateDQN().to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=1e-3)
        self.memory = deque(maxlen=50000)
        self.batch_size = 128
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.001
        self.epsilon_decay = 0.992
        self.target_update = 250
        self.steps = 0

    def select_move(self, candidates):
        if random.random() < self.epsilon:
            return random.choice(candidates)

        # Evaluate all candidate afterstates in a single fast batch
        feats = np.array([c["features"] for c in candidates], dtype=np.float32)
        feats_tensor = torch.tensor(feats, dtype=torch.float32, device=device)

        with torch.no_grad():
            values = self.policy_net(feats_tensor)
            # Immediate reward + discounted future value of the afterstate
            immediate_rewards = torch.tensor(
                [-10.0 if c["done"] else (1.0 + (c["lines"] ** 2) * 10.0) for c in candidates],
                dtype=torch.float32,
                device=device
            )
            scores = immediate_rewards + self.gamma * values * torch.tensor(
                [0.0 if c["done"] else 1.0 for c in candidates],
                dtype=torch.float32,
                device=device
            )
            best_idx = int(scores.argmax().item())

        return candidates[best_idx]

    def remember(self, afterstate_features, reward, next_candidates, done):
        next_feats = (
            np.zeros((1, 6), dtype=np.float32)
            if done or len(next_candidates) == 0
            else np.array([c["features"] for c in next_candidates], dtype=np.float32)
        )
        next_imm_rewards = (
            np.zeros(1, dtype=np.float32)
            if done or len(next_candidates) == 0
            else np.array(
                [-10.0 if c["done"] else (1.0 + (c["lines"] ** 2) * 10.0) for c in next_candidates],
                dtype=np.float32
            )
        )
        next_dones = (
            np.ones(1, dtype=bool)
            if done or len(next_candidates) == 0
            else np.array([c["done"] for c in next_candidates], dtype=bool)
        )
        self.memory.append((afterstate_features, reward, next_feats, next_imm_rewards, next_dones, done))

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return None

        batch = random.sample(self.memory, self.batch_size)
        states, rewards, next_feats_list, next_imm_list, next_dones_list, dones = zip(*batch)

        states_t = torch.tensor(np.array(states), dtype=torch.float32, device=device)
        current_v = self.policy_net(states_t)

        # Compute Bellman target: max over next piece's possible afterstates
        targets = []
        with torch.no_grad():
            for nf, ni, nd, d in zip(next_feats_list, next_imm_list, next_dones_list, dones):
                if d:
                    targets.append(0.0)
                else:
                    nf_t = torch.tensor(nf, dtype=torch.float32, device=device)
                    ni_t = torch.tensor(ni, dtype=torch.float32, device=device)
                    nd_t = torch.tensor(nd, dtype=torch.float32, device=device)

                    # Double DQN: policy picks best next afterstate, target evaluates it
                    policy_next_v = self.policy_net(nf_t)
                    policy_scores = ni_t + self.gamma * policy_next_v * (1.0 - nd_t)
                    best_next_idx = policy_scores.argmax()

                    target_next_v = self.target_net(nf_t)[best_next_idx]
                    best_target = ni_t[best_next_idx] + self.gamma * target_next_v * (1.0 - nd_t[best_next_idx])
                    targets.append(best_target.item())

        target_v = torch.tensor(targets, dtype=torch.float32, device=device)
        loss = nn.functional.smooth_l1_loss(current_v, target_v)

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 5.0)
        self.optimizer.step()

        self.steps += 1
        if self.steps % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return loss.item()

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [6]:
env = TetrisAfterstateEnv()
agent = AfterstateAgent()

episodes = 800
max_pieces = 2000
pieces_training = 4
scores = []
pieces_history = []
lines_history = []
loss_history = []

for episode in range(episodes):
    candidates = env.reset()
    total_reward = 0
    total_loss = []

    for step in range(max_pieces):
        if not candidates:
            break

        chosen = agent.select_move(candidates)
        next_candidates, reward, done = env.step(chosen)

        agent.remember(chosen["features"], reward, next_candidates, done)
        
        if env.pieces_placed % pieces_training == 0:
            loss = agent.train_step()
            if loss is not None:
                total_loss.append(loss)

        candidates = next_candidates
        total_reward += reward

        if done:
            break

    agent.decay_epsilon()
    scores.append(total_reward)
    pieces_history.append(env.pieces_placed)
    lines_history.append(env.rows_cleared)
    if total_loss:
        loss_history.append(np.mean(total_loss))

    if (episode + 1) % 10 == 0:
        avg_lines = np.mean(lines_history[-10:])
        print(
            f"Ep {episode + 1:3d}/{episodes} | "
            f"Pieces: {env.pieces_placed:4d} | "
            f"Lines: {env.rows_cleared:4d} | "
            f"Reward: {total_reward} | "
            f"10-Ep Avg Lines: {avg_lines:6.1f} | "
            f"Eps: {agent.epsilon:.3f}"
        )

Ep  10/800 | Pieces:   18 | Lines:    0 | Reward: 7.0 | 10-Ep Avg Lines:    0.0 | Eps: 0.923
Ep  20/800 | Pieces:   19 | Lines:    0 | Reward: 8.0 | 10-Ep Avg Lines:    0.0 | Eps: 0.852
Ep  30/800 | Pieces:   23 | Lines:    0 | Reward: 12.0 | 10-Ep Avg Lines:    0.0 | Eps: 0.786
Ep  40/800 | Pieces:   23 | Lines:    0 | Reward: 12.0 | 10-Ep Avg Lines:    0.0 | Eps: 0.725
Ep  50/800 | Pieces:   28 | Lines:    0 | Reward: 17.0 | 10-Ep Avg Lines:    0.1 | Eps: 0.669
Ep  60/800 | Pieces:   27 | Lines:    0 | Reward: 16.0 | 10-Ep Avg Lines:    0.1 | Eps: 0.618
Ep  70/800 | Pieces:   24 | Lines:    0 | Reward: 13.0 | 10-Ep Avg Lines:    0.2 | Eps: 0.570
Ep  80/800 | Pieces:   22 | Lines:    0 | Reward: 11.0 | 10-Ep Avg Lines:    0.2 | Eps: 0.526
Ep  90/800 | Pieces:   28 | Lines:    1 | Reward: 27.0 | 10-Ep Avg Lines:    0.3 | Eps: 0.485
Ep 100/800 | Pieces:   17 | Lines:    0 | Reward: 6.0 | 10-Ep Avg Lines:    0.0 | Eps: 0.448
Ep 110/800 | Pieces:   28 | Lines:    0 | Reward: 17.0 | 10-Ep 

##### 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 

In [22]:
torch.save(agent.policy_net.state_dict(), "dqn_tetris.pth")
print("model saved")

model saved


# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken



# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [1]:
import sys
sys.path.insert(0, "src")
from game import Game

game = Game("student")
pieces, rows = game.run_no_visual()

print("Pieces placed:", pieces)
print("Rows cleared:", rows)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


/home/jupyter-250661/tetris_env/lib/python3.12/site-packages/torch/cuda/__init__.py:241: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


RuntimeError: Error(s) in loading state_dict for AfterstateDQN:
	size mismatch for network.0.weight: copying a param with shape torch.Size([256, 279]) from checkpoint, the shape in current model is torch.Size([64, 6]).
	size mismatch for network.0.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([64]).
	size mismatch for network.2.weight: copying a param with shape torch.Size([256, 256]) from checkpoint, the shape in current model is torch.Size([64, 64]).
	size mismatch for network.2.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([64]).
	size mismatch for network.4.weight: copying a param with shape torch.Size([40, 256]) from checkpoint, the shape in current model is torch.Size([1, 64]).
	size mismatch for network.4.bias: copying a param with shape torch.Size([40]) from checkpoint, the shape in current model is torch.Size([1]).

In [3]:
import shutil
import sys
import importlib
from pathlib import Path

root_dir = Path("/home/jupyter-250661/Project_6/tetris_a")
shutil.copy2(root_dir / "src" / "dqn_tetris.pth", root_dir / "dqn_tetris.pth")
print("Updated root dqn_tetris.pth with the new AfterstateDQN weights!")

sys.path.insert(0, str(root_dir / "src"))
import custom_model
import game
importlib.reload(custom_model)
importlib.reload(game)

g = game.Game("student")
pieces, rows = g.run_no_visual()

print("Pieces placed:", pieces)
print("Rows cleared:", rows)

Updated root dqn_tetris.pth with the new AfterstateDQN weights!
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 1
9 1
10 2
11 2
12 3
13 3
14 3
15 3
16 3
17 3
18 5
19 6
20 6
21 7
22 7
23 8
24 8
25 8
26 8
27 9
28 10
29 11
30 11
31 11
32 11
33 11
34 11
35 12
36 13
37 14
38 14
39 14
40 15
41 15
42 16
43 16
44 16
45 16
46 16
47 17
48 18
49 18
50 19
51 19
52 19
53 19
54 20
55 21
56 21
57 21
58 22
59 22
60 22
61 23
62 23
63 23
64 23
65 23
66 23
67 23
68 23
69 23
70 25
71 26
72 26
73 26
74 27
75 27
76 28
77 28
78 28
79 28
80 28
81 28
82 29
83 30
84 31
85 31
86 31
87 31
88 31
89 32
90 33
91 33
92 33
93 33
94 35
95 36
96 37
97 37
98 37
99 37
100 38
101 38
102 40
103 41
104 41
105 41
106 42
107 43
108 44
109 44
110 44
111 44
112 45
113 45
114 46
115 46
116 46
117 46
118 46
119 47
120 47
121 47
122 49
123 50
124 50
125 50
126 50
127 51
128 51
129 51
130 51
131 51
132 51
133 51
134 51
135 51
136 51
137 51
138 53
139 54
140 54
141 54
142 55
143 55
144 57
145 58
146 58
147 59
148 60
149 60
150 60
151 61
152 61
153 61
